In [ ]:


from pathlib import Path
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import trimesh
from trimesh.voxel import ops as voxel_ops

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
DATA_DIR = Path(r"data_3dmodel")
CACHE_DIR = DATA_DIR / "_voxel_cache_64"
CACHE_DIR.mkdir(exist_ok=True)

VOXEL_SIZE = 64
IMAGE_SIZE = 128

png_files = sorted(DATA_DIR.glob('*.png'))
pairs = []
for img_path in png_files:
    stl_path = img_path.with_suffix('.stl')
    if stl_path.exists():
        pairs.append((img_path, stl_path))

print(f'Found pairs: {len(pairs)}')
if len(pairs) == 0:
    raise RuntimeError('No .png + .stl pairs found in data_3dmodel')

In [ ]:
def load_mesh(mesh_path: Path) -> trimesh.Trimesh:
    mesh = trimesh.load(mesh_path, force='mesh')
    if isinstance(mesh, trimesh.Scene):
        if len(mesh.geometry) == 0:
            raise ValueError(f'Empty scene: {mesh_path}')
        mesh = trimesh.util.concatenate(list(mesh.geometry.values()))
    return mesh

def mesh_to_voxel_grid(mesh_path: Path, grid_size: int = 64, pad: int = 2) -> np.ndarray:
    mesh = load_mesh(mesh_path).copy()

    verts = mesh.vertices
    vmin = verts.min(axis=0)
    vmax = verts.max(axis=0)
    span = np.maximum(vmax - vmin, 1e-6)

    verts = (verts - vmin) / span.max()
    scale = grid_size - 1 - 2 * pad
    verts = verts * scale + pad
    mesh.vertices = verts

    vox = mesh.voxelized(pitch=1.0)
    vox = vox.fill()
    points = np.round(vox.points).astype(np.int32)

    grid = np.zeros((grid_size, grid_size, grid_size), dtype=np.float32)
    valid = (
        (points[:, 0] >= 0) & (points[:, 0] < grid_size) &
        (points[:, 1] >= 0) & (points[:, 1] < grid_size) &
        (points[:, 2] >= 0) & (points[:, 2] < grid_size)
    )
    points = points[valid]
    grid[points[:, 0], points[:, 1], points[:, 2]] = 1.0
    return grid

class ImageToVoxelDataset(Dataset):
    def __init__(self, pairs, voxel_size=64, image_size=128, cache_dir=None):
        self.pairs = pairs
        self.voxel_size = voxel_size
        self.image_size = image_size
        self.cache_dir = cache_dir

    def __len__(self):
        return len(self.pairs)

    def _cache_file(self, stl_path: Path) -> Path:
        name = stl_path.stem + f'_vox{self.voxel_size}.npy'
        return self.cache_dir / name

    def __getitem__(self, idx):
        img_path, stl_path = self.pairs[idx]

        img = Image.open(img_path).convert('L').resize((self.image_size, self.image_size))
        img = np.asarray(img, dtype=np.float32) / 255.0
        img = torch.from_numpy(img).unsqueeze(0)

        cache_path = self._cache_file(stl_path)
        if cache_path.exists():
            vox = np.load(cache_path)
        else:
            vox = mesh_to_voxel_grid(stl_path, grid_size=self.voxel_size)
            np.save(cache_path, vox)

        vox = torch.from_numpy(vox).unsqueeze(0)
        return img, vox


In [ ]:
class ConvBlock2D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UNetLikeImageToVoxel(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        self.enc1 = ConvBlock2D(in_ch, 32)
        self.enc2 = ConvBlock2D(32, 64)
        self.enc3 = ConvBlock2D(64, 128)
        self.enc4 = ConvBlock2D(128, 256)
        self.pool = nn.MaxPool2d(2)

        self.bridge = nn.Sequential(
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )

        self.skip1 = nn.Conv2d(32, 16, 1)
        self.skip2 = nn.Conv2d(64, 32, 1)
        self.skip3 = nn.Conv2d(128, 64, 1)
        self.skip4 = nn.Conv2d(256, 128, 1)

        self.up1 = nn.ConvTranspose3d(256, 128, 2, stride=2)
        self.dec1 = ConvBlock3D(128, 128)
        self.up2 = nn.ConvTranspose3d(128, 64, 2, stride=2)
        self.dec2 = ConvBlock3D(64, 64)
        self.up3 = nn.ConvTranspose3d(64, 32, 2, stride=2)
        self.dec3 = ConvBlock3D(32, 32)
        self.up4 = nn.ConvTranspose3d(32, 16, 2, stride=2)
        self.dec4 = ConvBlock3D(16, 16)
        self.out = nn.Conv3d(16, out_ch, 1)

    def to_3d_skip(self, feat2d, proj, size):
        x = proj(feat2d)
        x = F.adaptive_avg_pool2d(x, (size, size))
        x = x.unsqueeze(2).repeat(1, 1, size, 1, 1)
        return x

    def forward(self, x):
        f1 = self.enc1(x)
        f2 = self.enc2(self.pool(f1))
        f3 = self.enc3(self.pool(f2))
        f4 = self.enc4(self.pool(f3))

        b = self.bridge(f4)
        b = b.unsqueeze(2).repeat(1, 1, 4, 1, 1)

        x = self.up1(b)
        x = x + self.to_3d_skip(f4, self.skip4, 8)
        x = self.dec1(x)

        x = self.up2(x)
        x = x + self.to_3d_skip(f3, self.skip3, 16)
        x = self.dec2(x)

        x = self.up3(x)
        x = x + self.to_3d_skip(f2, self.skip2, 32)
        x = self.dec3(x)

        x = self.up4(x)
        x = x + self.to_3d_skip(f1, self.skip1, 64)
        x = self.dec4(x)

        return self.out(x)


In [ ]:
def dice_loss_with_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    num = 2 * (probs * targets).sum(dim=(1,2,3,4))
    den = probs.sum(dim=(1,2,3,4)) + targets.sum(dim=(1,2,3,4)) + eps
    return 1 - (num / den).mean()

def combined_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    dsc = dice_loss_with_logits(logits, targets)
    return 0.6 * bce + 0.4 * dsc

indices = list(range(len(pairs)))
random.shuffle(indices)
split = int(0.9 * len(indices))
train_idx, val_idx = indices[:split], indices[split:]

train_pairs = [pairs[i] for i in train_idx]
val_pairs = [pairs[i] for i in val_idx]

train_ds = ImageToVoxelDataset(train_pairs, voxel_size=VOXEL_SIZE, image_size=IMAGE_SIZE, cache_dir=CACHE_DIR)
val_ds = ImageToVoxelDataset(val_pairs, voxel_size=VOXEL_SIZE, image_size=IMAGE_SIZE, cache_dir=CACHE_DIR)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=0)

model = UNetLikeImageToVoxel().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

In [ ]:
EPOCHS = 5
best_val = 1e9
best_path = DATA_DIR / 'best_image_to_voxel_unet_like.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, vox in train_loader:
        imgs = imgs.to(DEVICE)
        vox = vox.to(DEVICE)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = combined_loss(logits, vox)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, vox in val_loader:
            imgs = imgs.to(DEVICE)
            vox = vox.to(DEVICE)
            logits = model(imgs)
            val_loss += combined_loss(logits, vox).item()

    train_loss /= max(len(train_loader), 1)
    val_loss /= max(len(val_loader), 1)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), best_path)

    print(f'Epoch {epoch:02d}/{EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f}')



In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

img, gt = val_ds[0]
with torch.no_grad():
    pred_logits = model(img.unsqueeze(0).to(DEVICE))
    pred = (torch.sigmoid(pred_logits)[0, 0].cpu().numpy() > 0.4).astype(np.uint8)

gt = gt[0].numpy().astype(np.uint8)

mid = VOXEL_SIZE // 2
fig, ax = plt.subplots(2, 4, figsize=(14, 7))

ax[0, 0].imshow(img[0], cmap='gray')
ax[0, 0].set_title('Input image')
ax[0, 0].axis('off')

ax[0, 1].imshow(gt[mid], cmap='gray')
ax[0, 1].set_title('GT z-slice')
ax[0, 1].axis('off')
ax[0, 2].imshow(gt[:, mid, :], cmap='gray')
ax[0, 2].set_title('GT y-slice')
ax[0, 2].axis('off')
ax[0, 3].imshow(gt[:, :, mid], cmap='gray')
ax[0, 3].set_title('GT x-slice')
ax[0, 3].axis('off')

ax[1, 0].imshow(img[0], cmap='gray')
ax[1, 0].set_title('Input image')
ax[1, 0].axis('off')
ax[1, 1].imshow(pred[mid], cmap='gray')
ax[1, 1].set_title('Pred z-slice')
ax[1, 1].axis('off')
ax[1, 2].imshow(pred[:, mid, :], cmap='gray')
ax[1, 2].set_title('Pred y-slice')
ax[1, 2].axis('off')
ax[1, 3].imshow(pred[:, :, mid], cmap='gray')
ax[1, 3].set_title('Pred x-slice')
ax[1, 3].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
out_mesh = voxel_ops.matrix_to_marching_cubes(pred.astype(bool), pitch=1.0)
out_path = DATA_DIR / 'predicted_from_image.stl'
out_mesh.export(out_path)


In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

sample_id = 0
img, gt = val_ds[sample_id]
with torch.no_grad():
    pred_logits = model(img.unsqueeze(0).to(DEVICE))
    pred = (torch.sigmoid(pred_logits)[0, 0].cpu().numpy() > 0.4).astype(np.uint8)

gt = gt[0].numpy().astype(np.uint8)
img_np = img[0].numpy()

intersection = np.logical_and(pred == 1, gt == 1).sum()
union = np.logical_or(pred == 1, gt == 1).sum()
iou = intersection / (union + 1e-8)
dice = (2 * intersection) / (pred.sum() + gt.sum() + 1e-8)
print(f'Sample #{sample_id} | IoU={iou:.4f} | Dice={dice:.4f}')

def show_voxels(ax, vox, title, color='royalblue', step=2):
    # Downsample for faster rendering in notebooks.
    v = vox[::step, ::step, ::step].astype(bool)
    ax.voxels(v, facecolors=color, edgecolor='k', linewidth=0.05)
    ax.set_title(title)
    ax.set_axis_off()

mid = VOXEL_SIZE // 2
fig = plt.figure(figsize=(20, 9))
gs = fig.add_gridspec(2, 5, width_ratios=[1.2, 1, 1, 1, 1.4])

ax_input = fig.add_subplot(gs[:, 0])
ax_input.imshow(img_np, cmap='gray')
ax_input.set_title('Input image')
ax_input.axis('off')

ax_gt_z = fig.add_subplot(gs[0, 1])
ax_gt_z.imshow(gt[mid], cmap='gray')
ax_gt_z.set_title('GT z-slice')
ax_gt_z.axis('off')

ax_gt_y = fig.add_subplot(gs[0, 2])
ax_gt_y.imshow(gt[:, mid, :], cmap='gray')
ax_gt_y.set_title('GT y-slice')
ax_gt_y.axis('off')

ax_gt_x = fig.add_subplot(gs[0, 3])
ax_gt_x.imshow(gt[:, :, mid], cmap='gray')
ax_gt_x.set_title('GT x-slice')
ax_gt_x.axis('off')

ax_gt_3d = fig.add_subplot(gs[0, 4], projection='3d')
show_voxels(ax_gt_3d, gt, 'GT 3D voxels', color='seagreen', step=2)

ax_pr_z = fig.add_subplot(gs[1, 1])
ax_pr_z.imshow(pred[mid], cmap='gray')
ax_pr_z.set_title('Pred z-slice')
ax_pr_z.axis('off')

ax_pr_y = fig.add_subplot(gs[1, 2])
ax_pr_y.imshow(pred[:, mid, :], cmap='gray')
ax_pr_y.set_title('Pred y-slice')
ax_pr_y.axis('off')

ax_pr_x = fig.add_subplot(gs[1, 3])
ax_pr_x.imshow(pred[:, :, mid], cmap='gray')
ax_pr_x.set_title('Pred x-slice')
ax_pr_x.axis('off')

ax_pr_3d = fig.add_subplot(gs[1, 4], projection='3d')
show_voxels(ax_pr_3d, pred, 'Pred 3D voxels', color='royalblue', step=2)

plt.tight_layout()
plt.show()
